# 03 — Avaliação das regras de risco

Baseline transparente (0–3 pontos), divisão cronológica por `step` e avaliação retrospectiva. **Score ≥ 2 é o limiar de referência principal deste notebook.** Os limiares 1 e 3 foram executados separadamente como análise de sensibilidade; não são configurações concorrentes do fluxo principal. Os resultados são backtest exploratório: a EDA original já examinou rótulos de todos os períodos. Requer o CSV PaySim em `data/raw/`.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd()
if not (ROOT / 'src').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data.prepare_data import find_csv
from src.analysis.risk_rules import chronological_masks, fit_rule_thresholds, score_rules, alert_metrics, average_precision_stepwise
csv_path = find_csv(ROOT / 'data' / 'raw')
df = pd.read_csv(csv_path)
print(csv_path, df.shape)


In [ ]:
train_mask, valid_mask, test_mask = chronological_masks(df)
parts = []
for name, mask in [('treino', train_mask), ('validacao', valid_mask), ('teste_exploratorio', test_mask)]:
    part = df.loc[mask]
    parts.append({'periodo': name, 'step_inicio': part.step.min(), 'step_fim': part.step.max(), 'transacoes': len(part), 'fraudes': int(part.isFraud.sum()), 'prevalencia_pct': 100 * part.isFraud.mean()})
display(pd.DataFrame(parts))


## Construir score sem vazamento

In [ ]:
p95 = fit_rule_thresholds(df.loc[train_mask])
df['risk_score_baseline'] = score_rules(df, p95)
display(pd.Series(p95, name='P95 do valor no treino'))
print('Score 0–3; não usa isFraud, isFlaggedFraud ou saldos posteriores.')


## Validação — limiar de referência 2

In [ ]:
THRESHOLD = 2
valid = df.loc[valid_mask]
valid_metrics = alert_metrics(valid.isFraud, valid.risk_score_baseline, THRESHOLD)
valid_metrics['AP_degraus'] = average_precision_stepwise(valid.isFraud, valid.risk_score_baseline)
display(pd.DataFrame([valid_metrics]))


## Teste cronológico exploratório — mesmo limiar 2

In [ ]:
test = df.loc[test_mask]
test_metrics = alert_metrics(test.isFraud, test.risk_score_baseline, THRESHOLD)
test_metrics['AP_degraus'] = average_precision_stepwise(test.isFraud, test.risk_score_baseline)
display(pd.DataFrame([test_metrics]))


## Fila ilustrativa de alertas e justificativas

In [ ]:
alerts = valid.loc[valid.risk_score_baseline >= THRESHOLD].copy()
alerts['sinal_tipo'] = alerts.type.isin(['TRANSFER', 'CASH_OUT'])
alerts['sinal_valor_p95'] = alerts.amount.ge(alerts.type.map(p95))
alerts['sinal_saldo_esgotado'] = np.isclose(alerts.amount, alerts.oldbalanceOrg, rtol=1e-6, atol=.01)
alerts['motivos'] = alerts.apply(lambda r: '; '.join(x for ok, x in [(r.sinal_tipo, 'tipo TRANSFER/CASH_OUT'), (r.sinal_valor_p95, 'valor >= P95 no treino'), (r.sinal_saldo_esgotado, 'valor ~ saldo anterior')] if ok), axis=1)
display(alerts[['step', 'type', 'amount', 'risk_score_baseline', 'motivos']].head(20))
print(f'{len(alerts):,} alertas; fila ilustrativa, não operacional.')


## Interpretação

Na execução revisada, validação/limiar 2: 3.718 alertas, precisão 31,7%, recall 100% e 2.538 falsos positivos. Teste exploratório/limiar 2: 2.790 alertas, precisão 44,8%, recall 99,9%, 1.539 falsos positivos e 1 falso negativo. O limiar 2 é referência provisória, não política aprovada.

As execuções de limiar 1 e 3 foram análises de sensibilidade intencionais, executadas separadamente. Não fazem parte deste fluxo principal padronizado em 2; seus resultados e interpretação estão resumidos no relatório. Consulte `reports/intelligence_report.md`.
